# XGBoost: Predicting Hard-Drive Failure
**IST 707 Final Project**

This notebook trains an XGBoost model to predict whether a hard drive will fail in the next 30 days, using its SMART health readings from the last 30 days. It:

1. loads the training and calibration splits (the 2025 test split stays closed until final evaluation)
2. builds the model inputs, using the same feature rules as the random forest
3. trains a first XGBoost model with default settings and checks it on the calibration split
4. tunes XGBoost's settings with a randomized search over the **same** time-based folds as the random forest
5. saves the best model

**What XGBoost is:** it builds decision trees one after another, and each new tree focuses on the mistakes of the trees before it. It's usually one of the strongest methods on table-shaped data like ours, and it handles missing values by itself.

## 1. Libraries
Imports everything the notebook uses:
- **pandas / numpy:** loading and handling the data
- **scikit-learn:** the pipeline pieces (`Pipeline`, `ColumnTransformer`, `FunctionTransformer`, `OneHotEncoder`), the tuning tool (`RandomizedSearchCV`) and the scoring metrics
- **xgboost:** the `XGBClassifier` model. Install it once with `pip install xgboost`.
- **time / joblib:** timing the search and saving the final model to a file

`SimpleImputer` and `RandomForestClassifier` are also imported but not used in this notebook, because XGBoost doesn't need blanks filled.

In [1]:
import numpy as np
import pandas as pd
import time
import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss
)
from xgboost import XGBClassifier

## 2. Loading the Data
Loads the two splits used for training and checking. **The test split is not loaded.**

- **`modeling_train.parquet`:** Jan 2023 – Oct 2024. 169,827 drive-windows, 8,087 failures. Healthy windows were down-sampled to 20 per failure, so failures are 1 in 21.
- **`modeling_calib.parquet`:** Nov – Dec 2024. 297,009 drive-windows, 328 failures, at the real failure rate of about 1 in 905.

For each split the cell creates:
- **`X_..._raw`:** every column except the answer
- **`y_...`:** the answer, `label` (1 = the drive fails in the next 30 days)
- **`meta_...`:** serial number, window number and model. These are **not** model inputs; they're used to build the time-based folds.

In [2]:
train = pd.read_parquet("modeling_train.parquet")
calib = pd.read_parquet("modeling_calib.parquet")

X_train_raw, y_train = train.drop(columns="label"), train["label"].astype(int)
X_calib_raw, y_calib = calib.drop(columns="label"), calib["label"].astype(int)

meta_train = train[["serial_number", "window", "model"]].reset_index(drop=True)
meta_calib = calib[["serial_number", "window", "model"]].reset_index(drop=True)

print("Training shape:", X_train_raw.shape)
print("Calibration shape:", X_calib_raw.shape)

display(X_train_raw.head())
display(X_calib_raw.head())

display(y_train.head())
display(y_calib.head())

Training shape: (169827, 93)
Calibration shape: (297009, 93)


,serial_number,window,smart_10_normalized_first,smart_10_normalized_last,smart_10_raw_first,smart_10_raw_last,smart_12_normalized_first,smart_12_normalized_last,smart_12_raw_first,smart_12_raw_last,...,smart_3_raw_delta,smart_4_normalized_delta,smart_4_raw_delta,smart_5_normalized_delta,smart_5_raw_delta,smart_7_normalized_delta,smart_7_raw_delta,smart_9_normalized_delta,smart_9_raw_delta,drive_days_to_date
0,1050A093F97G,0,100.0,100.0,0.0,0.0,100.0,100.0,2.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,698.0,30
1,1050A0ANF97G,0,100.0,100.0,0.0,0.0,100.0,100.0,5.0,5.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,697.0,30
2,1080A06JF97G,0,100.0,100.0,0.0,0.0,100.0,100.0,6.0,6.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,697.0,30
3,1080A075F97G,0,100.0,100.0,0.0,0.0,100.0,100.0,7.0,7.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,681.0,30
4,1080A09LF97G,0,100.0,100.0,0.0,0.0,100.0,100.0,2.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,697.0,30


,serial_number,window,smart_10_normalized_first,smart_10_normalized_last,smart_10_raw_first,smart_10_raw_last,smart_12_normalized_first,smart_12_normalized_last,smart_12_raw_first,smart_12_raw_last,...,smart_3_raw_delta,smart_4_normalized_delta,smart_4_raw_delta,smart_5_normalized_delta,smart_5_raw_delta,smart_7_normalized_delta,smart_7_raw_delta,smart_9_normalized_delta,smart_9_raw_delta,drive_days_to_date
0,14A0A001F9RG,22,100.0,100.0,0.0,0.0,100.0,100.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,119.0,6
1,14B0A005F9RG,22,100.0,100.0,0.0,0.0,100.0,100.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,139.0,7
2,1PG6SGEV,22,100.0,100.0,0.0,0.0,100.0,100.0,8.0,8.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,78.0,5
3,1PG700ZV,22,100.0,100.0,0.0,0.0,100.0,100.0,10.0,10.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,94.0,5
4,1PG73SEV,22,100.0,100.0,0.0,0.0,100.0,100.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,171.0,8


0    0
1    0
2    0
3    0
4    0
Name: label, dtype: int64

0    0
1    0
2    0
3    0
4    0
Name: label, dtype: int64

## 3. Building the Model Inputs
Defines how the raw table is turned into model inputs. These are the **same rules as the random forest**, so the two models see the same information. This only sets rules; nothing is calculated from the data here.

- **`KEEP_RAW`:** the SMART attributes kept, as raw values only. The EDA showed the vendor-normalized values lose detail and partly encode the brand. S10 is left out because it is always 0.
- **`vendor()`:** reads the manufacturer (Seagate, Toshiba, WDC, HGST) from the model name.
- **`build_features()`:**
  - keeps the first, last and change values for each kept attribute
  - keeps S9 (power-on hours) as drive age
  - turns the unknown capacity of -1 into a blank (23 training rows)
  - keeps days observed and the vendor
- **`NUMERIC` / `CATEGORICAL`:** the lists of number columns and the one text column (vendor).
- **`features` / `vendor_onehot`:** wrap these rules so they can sit inside the pipeline. The one-hot encoder turns vendor into yes/no columns, using a fixed vendor list so train and calibration always get the same columns.

In [3]:
KEEP_RAW = [1, 3, 4, 5, 7, 12, 192, 193, 194, 197, 198, 199]   # raw only; S9 -> age, S10 always 0
VENDORS = ["Seagate", "Toshiba", "WDC", "HGST", "Other"]


def vendor(model):
    m = str(model)
    if m.startswith("ST"): return "Seagate"
    if m.startswith("TOSHIBA"): return "Toshiba"
    if m.startswith(("WDC", "WUH")): return "WDC"
    if m.startswith(("HGST", "Hitachi")): return "HGST"
    return "Other"

## This Function Keeps raw SMART values only. Drops useless columns. Calculates drive age through power-on hours and cleans the storage column.
def build_features(df):
    X = pd.DataFrame(index=df.index)
    for i in KEEP_RAW:
        for s in ("first", "last", "delta"):
            X[f"s{i}_{s}"] = df[f"smart_{i}_raw_{s}"].astype("float64")
    X["s9_last"] = df["smart_9_raw_last"].astype("float64")          
    X["capacity_tb"] = (df["capacity_bytes"] / 1e12).round(0).where(df["capacity_bytes"] > 0)   # -1 = unknown -> NaN
    X["days_observed"] = df["days_observed"].astype("float64")
    X["vendor"] = df["model"].map(vendor)
    return X

NUMERIC = [f"s{i}_{s}" for i in KEEP_RAW for s in ("first", "last", "delta")] + \
          ["s9_last", "capacity_tb", "days_observed"]

CATEGORICAL = ["vendor"]

features = FunctionTransformer(build_features)
vendor_onehot = OneHotEncoder(categories=[VENDORS], handle_unknown="ignore", sparse_output=False)

## 4. Preprocessing Pipeline
Chains the preprocessing steps into one object, `xgb_preprocess`:

1. **features:** applies `build_features` to the raw table
2. **columns:**
   - **numbers:** passed through **unchanged, blanks included**. XGBoost handles missing values itself: at every split it learns whether drives with a missing value should go left or right. The blanks come from specific drive models (S197 on two Toshiba models, S193 on one laptop model) and from unknown capacity.
   - **vendor:** one-hot encoded

**The difference from the random forest:** the random forest pipeline fills blanks with the training median and adds "was missing" columns, because scikit-learn's forest needs complete data. XGBoost doesn't, so filling them would only throw information away.

In [4]:
xgb_preprocess = Pipeline([
    ("features", features),
    ("columns", ColumnTransformer([
        ("numeric", "passthrough", NUMERIC),
        ("vendor", vendor_onehot, CATEGORICAL),
    ], verbose_feature_names_out=False)),
]).set_output(transform="pandas")

## 5. First XGBoost Model (Before Tuning)
Builds the full model, preprocessing followed by XGBoost. Only a few settings are fixed here; everything else uses XGBoost's defaults (100 trees, depth 6, learning rate 0.3):

- **`objective="binary:logistic"`:** a yes/no prediction that outputs a failure probability
- **`tree_method="hist"`:** the fast CPU method, which groups values into bins before searching for splits
- **`eval_metric="aucpr"`:** XGBoost's internal score is PR-AUC, our main metric
- **`n_jobs=-1`:** uses all CPU threads
- **`random_state=42`:** makes results repeatable

No extra weight is given to failures in this first model (`scale_pos_weight` keeps its default of 1). The tuning step tries 1, 5 and 20.

In [5]:
xgb_model = Pipeline([
    ("preprocess", xgb_preprocess),
    ("classifier", XGBClassifier(
        objective="binary:logistic",
        tree_method="hist",        # fast CPU method
        eval_metric="aucpr",       # PR-AUC, matches our main metric
        n_jobs=-1,                 # all 8 threads per model
        random_state=42,
    )),
])

### 5.1 Training the First Model
Fits the whole pipeline on the training split: the preprocessing builds the features, then XGBoost builds its trees one after another.

In [6]:
xgb_model.fit(X_train_raw, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](93,)","['serial_number','window','smart_10_normalized_first',..., 'smart_9_normalized_delta','smart_9_raw_delta','drive_days_to_date']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,93
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('features', ...), ('columns', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the trans

### 5.2 Checking the First Model on the Calibration Split
Predicts a failure probability for every calibration window, then turns those probabilities into yes/no predictions at thresholds from 0.0 to 1.0. At each threshold it records:

- **PR-AUC (AP):** how well the model *ranks* failing drives above healthy ones, over all thresholds. This is our main metric, and random guessing scores 0.0011.
- **ROC-AUC:** another ranking score (0.5 = coin flip, 1.0 = perfect). It looks good easily when failures are rare, so it's secondary.
- **Precision:** of the drives flagged, the share that actually failed
- **Recall:** of the drives that failed, the share that were flagged
- **F1:** one number balancing precision and recall
- **Brier score:** how close the probabilities are to what actually happened (lower is better)

PR-AUC, ROC-AUC and the Brier score don't depend on the threshold, which is why they're the same in every row. Only precision, recall and F1 change.

**Result (untuned XGBoost):**
- **Ranking:** PR-AUC is **0.0927**. That's slightly below the untuned random forest (0.0995), well above logistic regression (0.082) and far above the SMART 5 rule (0.0073) and random (0.0011). ROC-AUC is 0.883.
- **Thresholds:** at 0.9, precision is 14.3% and recall 24.7%, the best F1 (0.181). At the default 0.5, precision is only 6.3%.
- **Calibration:** the Brier score (0.0076) is better than the random forest's (0.0119), because without extra failure weighting the probabilities are less inflated. They're still too high for the real failure rate, so a threshold has to be chosen deliberately and the probabilities recalibrated later.

In [7]:
results=[]
y_prob = xgb_model.predict_proba(X_calib_raw)[:, 1]

# Default decision threshold
for x in [0.0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0]:
    y_pred = (y_prob >= x).astype(int)

    results.append({
        "Threshold": x,
        "Model": "XGBoost",
        "PR-AUC (AP)": average_precision_score(y_calib, y_prob),
        "ROC-AUC": roc_auc_score(y_calib, y_prob),
        "Precision": precision_score(
            y_calib, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_calib, y_pred, zero_division=0
        ),
        "F1": f1_score(y_calib, y_pred, zero_division=0),
        "Brier Score": brier_score_loss(y_calib, y_prob)
    })

results_df = pd.DataFrame(results)
display(results_df.round(4))

,Threshold,Model,PR-AUC (AP),ROC-AUC,Precision,Recall,F1,Brier Score
0,0.0,XGBoost,0.0927,0.8828,0.0011,1.0000,0.0022,0.0076
1,0.1,XGBoost,0.0927,0.8828,0.0225,0.7073,0.0435,0.0076
2,0.2,XGBoost,0.0927,0.8828,0.0342,0.6677,0.0651,0.0076
3,0.3,XGBoost,0.0927,0.8828,0.0439,0.6311,0.0820,0.0076
4,0.4,XGBoost,0.0927,0.8828,0.0539,0.6098,0.0990,0.0076
5,0.5,XGBoost,0.0927,0.8828,0.0625,0.5762,0.1128,0.0076
6,0.6,XGBoost,0.0927,0.8828,0.0730,0.5244,0.1282,0.0076
7,0.7,XGBoost,0.0927,0.8828,0.0876,0.4756,0.1480,0.0076
8,0.8,XGBoost,0.0927,0.8828,0.1060,0.3780,0.1656,0.0076
9,0.9,XGBoost,0.0927,0.8828,0.1426,0.2470,0.1808,0.0076


## 6. Tuning XGBoost
### 6.1 Search Setup and Time-Based Folds
Sets up three things for tuning:

- **`xgb_pipeline`:** the same preprocessing plus an XGBoost model whose settings will be searched
- **`param_distributions`:** the settings to try:
  - **`n_estimators`:** how many trees are built in sequence
  - **`learning_rate`:** how much each new tree corrects the previous ones. Smaller is more careful but needs more trees.
  - **`max_depth`:** how deep each tree can grow. Boosted trees are usually shallow.
  - **`min_child_weight`:** the minimum amount of data in a leaf, similar to the random forest's `min_samples_leaf`
  - **`subsample` / `colsample_bytree`:** the share of rows and of columns each tree sees, which adds variety and limits memorizing
  - **`gamma`:** the minimum improvement needed before a split is made
  - **`reg_lambda`:** a penalty that keeps leaf values small, to reduce overfitting
  - **`scale_pos_weight`:** extra weight on failures: 1 = none, 20 = matches train's 20:1 ratio

  Together that's 26,244 combinations.
- **`time_folds()`:** builds the **same 4 time-based folds as the random forest**, so both models are compared on the same months. Each fold trains on earlier months and validates on later months, with a one-window gap in between, because each label looks one window ahead.

The printout shows the 4 folds growing: training uses more months each time, and each validation block covers the next few months.

In [8]:
xgb_pipeline = Pipeline([
    ("preprocess", xgb_preprocess),
    ("classifier", XGBClassifier(
        objective="binary:logistic",
        tree_method="hist",        # fast CPU method
        eval_metric="aucpr",       # PR-AUC, matches our main metric
        n_jobs=-1,                 # all 8 threads per model
        random_state=42,
    )),
])

param_distributions = {
    "classifier__n_estimators": [300, 500, 800],
    "classifier__learning_rate": [0.03, 0.05, 0.1],     # how big each tree's correction is
    "classifier__max_depth": [3, 4, 6, 8],               # boosted trees are shallow
    "classifier__min_child_weight": [1, 5, 10],          # minimum weight in a leaf (like min_samples_leaf)
    "classifier__subsample": [0.7, 0.8, 1.0],            # share of rows each tree sees
    "classifier__colsample_bytree": [0.5, 0.7, 1.0],     # share of columns each tree sees
    "classifier__gamma": [0, 1, 5],                      # minimum gain needed to make a split
    "classifier__reg_lambda": [1, 5, 10],                # penalty that keeps leaf values small
    "classifier__scale_pos_weight": [1, 5, 20],          # extra weight on failures (train is 20:1)
}


## Creating a cross validation set.
def time_folds(meta, n_splits=4, gap=1):
    """Expanding-window folds by calendar window. Skips `gap` windows between train and validation,
    because a window's label looks one window ahead."""
    w = meta["window"].to_numpy()
    windows = np.sort(np.unique(w))
    blocks = np.array_split(windows, n_splits + 1)
    folds = []
    for k in range(1, n_splits + 1):
        valid_w = blocks[k]
        train_w = windows[windows < valid_w.min() - gap]
        folds.append((np.flatnonzero(np.isin(w, train_w)), np.flatnonzero(np.isin(w, valid_w))))
    return folds

cv_folds = time_folds(meta_train)
for k, (tr_idx, va_idx) in enumerate(cv_folds):
    print(f"fold {k}: train {len(tr_idx):,} rows -> validate {len(va_idx):,} rows")


fold 0: train 28,508 rows -> validate 37,027 rows
fold 1: train 64,596 rows -> validate 31,106 rows
fold 2: train 95,671 rows -> validate 32,768 rows
fold 3: train 128,057 rows -> validate 33,501 rows


### 6.2 Randomized Search, Results and Saving the Best Model
- **Randomized search:** tries **20 random combinations**. Each is trained once per fold, giving **80 fits**. `n_jobs=1` trains one model at a time, because each model already uses every CPU thread.
- **Scoring:** every fit is scored on four metrics:
  - **PR-AUC** (`average_precision`), which picks the winner (`refit="pr_auc"`)
  - **precision, recall and F1**, at XGBoost's default cut-off of 0.5
- **Refit:** the best combination by PR-AUC is retrained on all of the training data.
- **Results table:** each metric's average over the 4 folds (`PR_AUC`, `Precision`, `Recall`, `F1`), how much it varies across folds (`..._std`), the training time, the settings tried, and a rank (1 = best PR-AUC). The table is saved to `xgb_search_results_ranked.csv`.
- **Save:** the best model is saved to `xgb_best.joblib`, so it can be reloaded without retraining.

These scores come from training folds, where failures are 1 in 21, so they're much higher than the same metrics on the calibration split. Use them only to compare settings with each other. Precision, recall and F1 also depend on the 0.5 cut-off, so PR-AUC is the fairer comparison.

**This cell has not been run yet.** Its results will appear here once it is. Expect roughly 15–60 minutes.

In [9]:
search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=param_distributions,
    n_iter=20,
    scoring={
        "precision": "precision",
        "recall": "recall",
        "f1": "f1",
        "pr_auc": "average_precision",
    },
    cv=cv_folds,                   # same time-based folds as the random forest
    n_jobs=1,                      # one model at a time; each uses all threads
    refit="pr_auc",
    random_state=42,
    verbose=2,
    return_train_score=False,
)


start = time.time()
search.fit(X_train_raw, y_train)
print(f"\nSearch took {(time.time() - start) / 60:.1f} minutes")

cv_results = pd.DataFrame(search.cv_results_)

# All four metrics, averaged across the CV folds (column names follow the scoring keys)
results = pd.DataFrame({
    "PR_AUC":         cv_results["mean_test_pr_auc"],
    "PR_AUC_std":     cv_results["std_test_pr_auc"],
    "Precision":      cv_results["mean_test_precision"],
    "Precision_std":  cv_results["std_test_precision"],
    "Recall":         cv_results["mean_test_recall"],
    "Recall_std":     cv_results["std_test_recall"],
    "F1":             cv_results["mean_test_f1"],
    "F1_std":         cv_results["std_test_f1"],
    "mean_fit_time":  cv_results["mean_fit_time"],
})

for col in cv_results.columns:
    if col.startswith("param_"):
        results[col] = cv_results[col]

# Rank by PR-AUC (the same ranking the search used to pick the best model)
results["PR_AUC_rank"] = cv_results["rank_test_pr_auc"]
results = results.sort_values("PR_AUC_rank").reset_index(drop=True)

print("Best PR-AUC (time-CV on train):", round(search.best_score_, 4))
print("Best parameters:")
for k, v in search.best_params_.items():
    print(f"  {k}: {v}")


print("\nTop 10 models ranked by PR-AUC:")
display(results.head(10))
results.to_csv("xgb_search_results_ranked.csv", index=False)

# ---------------------------------------------------------------- 6. Save the best model
best_xgb = search.best_estimator_
joblib.dump(best_xgb, "xgb_best.joblib")


Fitting 4 folds for each of 20 candidates, totalling 80 fits
[CV] END classifier__colsample_bytree=1.0, classifier__gamma=5, classifier__learning_rate=0.03, classifier__max_depth=4, classifier__min_child_weight=5, classifier__n_estimators=300, classifier__reg_lambda=1, classifier__scale_pos_weight=1, classifier__subsample=1.0; total time=   0.9s
[CV] END classifier__colsample_bytree=1.0, classifier__gamma=5, classifier__learning_rate=0.03, classifier__max_depth=4, classifier__min_child_weight=5, classifier__n_estimators=300, classifier__reg_lambda=1, classifier__scale_pos_weight=1, classifier__subsample=1.0; total time=   1.5s
[CV] END classifier__colsample_bytree=1.0, classifier__gamma=5, classifier__learning_rate=0.03, classifier__max_depth=4, classifier__min_child_weight=5, classifier__n_estimators=300, classifier__reg_lambda=1, classifier__scale_pos_weight=1, classifier__subsample=1.0; total time=   2.0s
[CV] END classifier__colsample_bytree=1.0, classifier__gamma=5, classifier__le

,PR_AUC,PR_AUC_std,Precision,Precision_std,Recall,Recall_std,F1,F1_std,mean_fit_time,param_classifier__subsample,param_classifier__scale_pos_weight,param_classifier__reg_lambda,param_classifier__n_estimators,param_classifier__min_child_weight,param_classifier__max_depth,param_classifier__learning_rate,param_classifier__gamma,param_classifier__colsample_bytree,PR_AUC_rank
0,0.595183,0.030466,0.744773,0.016985,0.484026,0.030201,0.586494,0.027449,1.957569,0.7,1,1,300,1,4,0.05,5,0.7,1
1,0.592214,0.032883,0.745720,0.015872,0.485790,0.029614,0.588097,0.026564,2.294284,1.0,1,5,300,5,6,0.05,1,1.0,2
2,0.590543,0.029190,0.746649,0.010546,0.480387,0.032035,0.584161,0.026320,1.663272,1.0,1,1,300,5,4,0.03,5,1.0,3
3,0.588849,0.028980,0.740423,0.012468,0.480609,0.028893,0.582566,0.024785,4.055853,0.8,1,1,500,10,4,0.03,5,0.5,4
4,0.588802,0.034905,0.598725,0.024056,0.602228,0.045638,0.600045,0.032941,4.805528,1.0,5,10,500,5,8,0.03,0,0.5,5
5,0.588738,0.034948,0.577237,0.018911,0.615972,0.044554,0.595461,0.028454,5.616486,0.7,5,1,800,1,4,0.03,1,0.7,6
6,0.588335,0.029214,0.742745,0.009286,0.483003,0.028003,0.585028,0.023304,1.743010,1.0,1,1,300,5,4,0.10,5,0.7,7
7,0.587372,0.031406,0.744347,0.013599,0.472713,0.027633,0.577978,0.024716,2.605395,1.0,1,1,500,10,3,0.05,1,1.0,8
8,0.587088,0.035500,0.577207,0.024864,0.619791,0.041880,0.597501,0.031343,5.701103,0.7,5,5,800,10,4,0.05,5,0.7,9
9,0.585834,0.036038,0.737533,0.019020,0.479109,0.029045,0.580702,0.027043,5.521930,1.0,1,5,800,10,6,0.05,0,0.5,10


['xgb_best.joblib']